# 11장: 몬테카를로 시뮬레이션 (Monte Carlo Simulation)

## 학습 목표
1. 점 추정의 한계를 이해하고 확률적 추정의 필요성을 설명한다
2. 주요 확률 분포(정규, 삼각, 균등, 로그정규)의 특성과 적용 상황을 구분한다
3. 몬테카를로 시뮬레이션으로 매출/비용/이익의 분포를 도출한다
4. 민감도 분석(토네이도 차트)으로 핵심 리스크 변수를 식별한다
5. 시나리오 플래닝(10장)과 시뮬레이션을 통합하여 강건한 전략을 평가한다

## 강의 구조
| 시간 | 내용 | 활동 |
|------|------|------|
| Part 1 | 점 추정의 한계, 확률 분포 | 이론 + 시각화 + 조사 과제 |
| Part 2 | 사업 시뮬레이션, 민감도 분석 | 이론 + 시각화 + 조사 과제 |
| 휴식 | — | 15분 |
| Part 3 | 시나리오 × 시뮬레이션 통합 | 이론 + 시각화 + 조사 과제 |
| Part 4 | 실습: 사업 계획 시뮬레이션 | 코드 작성 |
| Part 5 | 실습: NPV 리스크 분석 | 코드 작성 |

In [ ]:
# ============================================================
# 환경설정 및 라이브러리 로드
# ============================================================
# 처음 사용 시 아래 명령을 터미널에서 먼저 실행하세요:
#   python setup_env.py
# 그 후 VSCode 우측 상단에서 커널을 'AI 기획 강의 (Python 3)'으로 선택하세요.
# ============================================================

import sys

# 패키지 확인
_required = ["numpy", "pandas", "plotly"]
_missing = []
for _pkg in _required:
    try:
        __import__(_pkg)
    except ImportError:
        _missing.append(_pkg)
if _missing:
    print(f"❌ 누락된 패키지: {', '.join(_missing)}")
    print("   터미널에서 다음 명령을 실행하세요: python setup_env.py")
    raise SystemExit(1)

# 라이브러리 임포트
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ 라이브러리 로드 완료!")

---
# Part 1: 점 추정의 한계와 확률적 사고

## 1.1 점 추정 vs 확률적 추정

기획 보고서에서 흔히 보는 표현: **"예상 매출 1억 원"**

이 숫자는 직관적이지만, 핵심 질문에 답하지 못한다:
- 이 숫자가 얼마나 **확실**한가?
- **최악의 경우** 얼마인가?
- **목표 달성 확률**은 얼마인가?

| 구분 | 점 추정 | 확률적 추정 |
|------|---------|------------|
| 표현 | "매출 1억 원 예상" | "8천만~1.2억 원 (90% CI)" |
| 불확실성 | 숨김 | 명시적 표현 |
| 리스크 평가 | 불가능 | VaR, 손실 확률 계산 가능 |
| 의사결정 | 단일 시나리오 | 확률 기반 의사결정 |

## 1.2 몬테카를로 시뮬레이션이란?

**몬테카를로 시뮬레이션**은 난수(random number)를 반복 생성하여 결과의 확률 분포를 구하는 기법이다. 이름은 모나코의 도박 도시 Monte Carlo에서 유래했다.

**기본 원리:**
1. 입력 변수를 **확률 분포**로 정의
2. 분포에서 **난수를 생성**하여 모델에 대입
3. **수천~수만 번 반복**하여 결과의 분포를 얻음
4. 분포에서 **핵심 지표**(평균, VaR, 성공 확률 등) 도출

## 1.3 확률 분포의 선택

입력 변수에 적합한 분포를 선택하는 것이 시뮬레이션의 핵심이다:

| 분포 | 특징 | 적용 상황 | NumPy 함수 |
|------|------|----------|-----------|
| **정규분포** | 좌우 대칭, 평균 중심 | 과거 데이터 충분 | `np.random.normal(μ, σ, n)` |
| **삼각분포** | 최소/최빈/최대 지정 | 전문가 추정 | `np.random.triangular(min, mode, max, n)` |
| **균등분포** | 모든 값 동일 확률 | 정보 부족 | `np.random.uniform(min, max, n)` |
| **로그정규분포** | 양수, 오른쪽 꼬리 | 비용, 시간 | `np.random.lognormal(μ, σ, n)` |

> **실무 팁**: 데이터가 없을 때는 **삼각분포**가 가장 실용적이다. 전문가에게 "최악?", "최선?", "가장 가능성 높은 값?"만 물으면 된다.

In [ ]:
# 4가지 주요 확률 분포 비교
n = 10000

distributions = {
    'Normal (μ=100, σ=20)': np.random.normal(100, 20, n),
    'Triangular (60, 100, 150)': np.random.triangular(60, 100, 150, n),
    'Uniform (50, 150)': np.random.uniform(50, 150, n),
    'Lognormal (μ=4.5, σ=0.3)': np.random.lognormal(4.5, 0.3, n)
}

colors = ['#4C78A8', '#F58518', '#54A24B', '#E45756']

fig = make_subplots(rows=2, cols=2, subplot_titles=list(distributions.keys()))

for i, (name, data) in enumerate(distributions.items()):
    row, col = i // 2 + 1, i % 2 + 1
    fig.add_trace(
        go.Histogram(x=data, nbinsx=50, marker_color=colors[i],
                     opacity=0.75, name=name, showlegend=False),
        row=row, col=col
    )
    # Add mean line
    fig.add_vline(x=np.mean(data), line_dash="dash", line_color="red",
                  row=row, col=col)

fig.update_layout(
    title='Comparison of Key Probability Distributions (n=10,000)',
    height=500, width=800
)
fig.show()

# 통계 요약
print("📊 분포별 통계:")
print(f"{'Distribution':<35} {'Mean':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 70)
for name, data in distributions.items():
    print(f"{name:<35} {np.mean(data):>8.1f} {np.std(data):>8.1f} {np.min(data):>8.1f} {np.max(data):>8.1f}")

print("\n💡 삼각분포는 전문가 추정에, 로그정규분포는 양수 값(비용, 시간)에 적합하다.")

### 📝 이론 과제 11-1 (15분)

#### 과제: 확률 분포와 몬테카를로 시뮬레이션

**질문:**
1. **분포 선택**: 다음 각 변수에 어떤 확률 분포가 적합한지 선택하고 이유를 설명하시오. (각 2문장)
   - (a) 신규 음식점의 월 매출
   - (b) 소프트웨어 프로젝트의 완료 기간
   - (c) 완전히 새로운 시장의 규모 (정보가 거의 없는 경우)
2. **MC 적용 사례**: 몬테카를로 시뮬레이션이 실제로 활용된 분야(예: 금융, 제약, 건설 등) 1가지를 조사하고, 어떤 문제를 해결했는지 설명하시오. (3-4문장)
3. **시뮬레이션 횟수**: 시뮬레이션 횟수를 늘리면 결과가 어떻게 변하는가? "수렴(convergence)"의 의미를 설명하시오. (2-3문장)

**조사 키워드:**
- "Monte Carlo simulation application"
- "triangular distribution expert elicitation"
- "simulation convergence law of large numbers"

**제출:** 아래 마크다운 셀에 **직접 타이핑**

### ✅ 과제 11-1 제출란

**학번:** ___________
**이름:** ___________

#### 1. 분포 선택

- **(a) 신규 음식점 월 매출:** (여기에 작성)
- **(b) 소프트웨어 프로젝트 완료 기간:** (여기에 작성)
- **(c) 새로운 시장 규모:** (여기에 작성)

#### 2. MC 시뮬레이션 적용 사례

(여기에 작성)

#### 3. 시뮬레이션 수렴

(여기에 작성)

---
# Part 2: 사업 계획 시뮬레이션

## 2.1 매출/비용 시뮬레이션

전자제품 제조업체 A사의 신규 사업을 분석한다. 입력 변수 5개를 삼각분포로 정의한다:

| 변수 | 최소 | 최빈 | 최대 | 단위 |
|------|------|------|------|------|
| 시장 규모 | 800 | 1,000 | 1,200 | 억 원 |
| 시장 점유율 | 5% | 10% | 15% | — |
| 변동비율 | 40% | 50% | 60% | — |
| 고정비 | 30 | 40 | 50 | 억 원 |

**모델:**
Revenue = Market Size × Market Share
Profit = Revenue - (Revenue × Variable Cost Ratio + Fixed Cost)

In [ ]:
# 사업 계획 몬테카를로 시뮬레이션
n_sim = 10000

# 입력 변수 (삼각분포)
market_size = np.random.triangular(800, 1000, 1200, n_sim)       # 억원
market_share = np.random.triangular(0.05, 0.10, 0.15, n_sim)     # 비율
var_cost_ratio = np.random.triangular(0.40, 0.50, 0.60, n_sim)   # 비율
fixed_cost = np.random.triangular(30, 40, 50, n_sim)             # 억원

# 모델 계산
revenue = market_size * market_share
variable_cost = revenue * var_cost_ratio
profit = revenue - variable_cost - fixed_cost

# 결과 시각화: 이익 분포
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=profit, nbinsx=60,
    marker_color='#4C78A8', opacity=0.75,
    name='Profit Distribution'
))

# 주요 지표 라인
fig.add_vline(x=0, line_dash="solid", line_color="red", line_width=2,
              annotation_text="Breakeven", annotation_position="top")
fig.add_vline(x=np.mean(profit), line_dash="dash", line_color="green", line_width=2,
              annotation_text=f"Mean={np.mean(profit):.1f}", annotation_position="top")
fig.add_vline(x=np.percentile(profit, 5), line_dash="dot", line_color="orange",
              annotation_text=f"5th={np.percentile(profit,5):.1f}", annotation_position="bottom left")
fig.add_vline(x=np.percentile(profit, 95), line_dash="dot", line_color="orange",
              annotation_text=f"95th={np.percentile(profit,95):.1f}", annotation_position="bottom right")

fig.update_layout(
    title=f'Business Profit Distribution (n={n_sim:,} simulations)',
    xaxis_title='Profit (100M KRW)',
    yaxis_title='Frequency',
    height=450, width=750
)
fig.show()

# 핵심 통계
print("📊 시뮬레이션 결과 (10,000회):")
print(f"  평균 이익: {np.mean(profit):.1f}억 원")
print(f"  표준편차: {np.std(profit):.1f}억 원")
print(f"  90% 신뢰구간: [{np.percentile(profit,5):.1f}, {np.percentile(profit,95):.1f}]억 원")
print(f"  흑자 확률: {np.mean(profit > 0)*100:.1f}%")
print(f"  적자 확률: {np.mean(profit < 0)*100:.1f}%")
print(f"\n💡 평균 이익은 약 10억 원이지만, 적자 확률이 ~22%에 달한다.")
print(f"   '예상 이익 10억 원'만 보고 의사결정하면 리스크를 과소평가하게 된다.")

In [ ]:
# 민감도 분석 (토네이도 차트)
variables = {
    'Market Size': market_size,
    'Market Share': market_share,
    'Variable Cost Ratio': var_cost_ratio,
    'Fixed Cost': fixed_cost
}

sensitivities = []
for var_name, var_data in variables.items():
    low_mask = var_data <= np.percentile(var_data, 10)
    high_mask = var_data >= np.percentile(var_data, 90)
    low_profit = np.mean(profit[low_mask])
    high_profit = np.mean(profit[high_mask])
    swing = abs(high_profit - low_profit)
    sensitivities.append({
        'name': var_name,
        'low': low_profit,
        'high': high_profit,
        'swing': swing
    })

# 변동폭 순으로 정렬
sensitivities.sort(key=lambda x: x['swing'])
mean_profit = np.mean(profit)

fig = go.Figure()

for s in sensitivities:
    fig.add_trace(go.Bar(
        y=[s['name']], x=[s['low'] - mean_profit],
        orientation='h', marker_color='#E45756',
        name='Low (10th pct)', showlegend=False,
        text=f"{s['low']:.1f}", textposition='outside',
        hovertemplate=f"When {s['name']} is LOW: Profit={s['low']:.1f}"
    ))
    fig.add_trace(go.Bar(
        y=[s['name']], x=[s['high'] - mean_profit],
        orientation='h', marker_color='#54A24B',
        name='High (90th pct)', showlegend=False,
        text=f"{s['high']:.1f}", textposition='outside',
        hovertemplate=f"When {s['name']} is HIGH: Profit={s['high']:.1f}"
    ))

fig.add_vline(x=0, line_color="black", line_width=1)

fig.update_layout(
    title=f'Tornado Chart: Sensitivity Analysis (Base Profit = {mean_profit:.1f})',
    xaxis_title='Deviation from Mean Profit (100M KRW)',
    height=350, width=750,
    barmode='overlay',
    showlegend=False
)
fig.show()

# 민감도 순위 (변동폭 내림차순)
print("📊 민감도 순위 (변동폭 기준):")
for i, s in enumerate(reversed(sensitivities), 1):
    print(f"  {i}. {s['name']}: 변동폭 {s['swing']:.1f}억 원")
print(f"\n💡 시장 점유율(Market Share)이 이익에 가장 큰 영향을 미친다.")
print(f"   → 마케팅 투자와 차별화 전략이 최우선 과제이다.")

### 📝 이론 과제 11-2 (15분)

#### 과제: 리스크 지표와 민감도 분석

**질문:**
1. **VaR와 CVaR**: Value at Risk(VaR)와 Conditional VaR(CVaR)의 차이를 설명하시오. 위 시뮬레이션에서 VaR(5%)는 무엇을 의미하는가? (3문장)
2. **토네이도 차트 해석**: 위 토네이도 차트에서 "시장 점유율이 가장 민감한 변수"라는 것은 실무적으로 어떤 시사점을 주는가? (2-3문장)
3. **Garbage In, Garbage Out**: 시뮬레이션의 핵심 한계인 "GIGO" 원칙을 설명하고, 입력 분포의 품질을 높이기 위한 방법 2가지를 제시하시오.

**조사 키워드:**
- "Value at Risk VaR explained"
- "tornado chart sensitivity analysis"
- "Monte Carlo simulation limitations GIGO"

**제출:** 아래 마크다운 셀에 **직접 타이핑**

### ✅ 과제 11-2 제출란

**학번:** ___________
**이름:** ___________

#### 1. VaR와 CVaR

(여기에 작성)

#### 2. 토네이도 차트의 실무적 시사점

(여기에 작성)

#### 3. GIGO 원칙과 입력 품질 향상 방법

(여기에 작성)

---
# ― 휴식 (15분) ―
---

# Part 3: 시나리오 × 시뮬레이션 통합

## 3.1 시나리오별 시뮬레이션

10장에서 개발한 4개 시나리오를 시뮬레이션과 통합한다. 각 시나리오는 **시장 성장률**과 **마진율**이 다르다:

| 시나리오 | 확률 | 시장 성장 배수 | 마진율 |
|----------|------|--------------|--------|
| 🟢 Green Acceleration | 25% | 2.0× | 30% |
| 🔵 Regulation-Driven | 20% | 1.4× | 20% |
| 🟡 Market-Driven | 30% | 1.8× | 25% |
| 🔴 Gradual Transition | 25% | 1.1× | 15% |

시나리오 플래닝이 **"어떤 미래가 올 수 있는가?"**에 답한다면,
몬테카를로 시뮬레이션은 **"그 미래에서 결과가 어떤 범위에 있을 확률은 얼마인가?"**에 답한다.

In [ ]:
# 시나리오 × 시뮬레이션 통합
scenarios = {
    'Green Acceleration':  {'growth': 2.0, 'margin': 0.30, 'prob': 0.25, 'color': '#54A24B'},
    'Regulation-Driven':   {'growth': 1.4, 'margin': 0.20, 'prob': 0.20, 'color': '#4C78A8'},
    'Market-Driven':       {'growth': 1.8, 'margin': 0.25, 'prob': 0.30, 'color': '#F58518'},
    'Gradual Transition':  {'growth': 1.1, 'margin': 0.15, 'prob': 0.25, 'color': '#E45756'}
}

n_sim = 10000
base_revenue = 100  # 기준 매출 100억 원
fixed_cost_base = 40  # 기준 고정비 40억 원

fig = go.Figure()
scenario_results = {}

for name, params in scenarios.items():
    # 시나리오별 매출 시뮬레이션
    growth = np.random.triangular(
        params['growth'] * 0.8, params['growth'], params['growth'] * 1.2, n_sim
    )
    margin = np.random.triangular(
        params['margin'] * 0.7, params['margin'], params['margin'] * 1.3, n_sim
    )
    
    revenue = base_revenue * growth
    profit = revenue * margin - fixed_cost_base
    
    scenario_results[name] = {
        'profit': profit,
        'mean': np.mean(profit),
        'std': np.std(profit),
        'prob_positive': np.mean(profit > 0) * 100,
        'p5': np.percentile(profit, 5),
        'p95': np.percentile(profit, 95)
    }
    
    fig.add_trace(go.Histogram(
        x=profit, nbinsx=50, opacity=0.5,
        marker_color=params['color'],
        name=f"{name} (P={params['prob']:.0%})"
    ))

fig.add_vline(x=0, line_dash="solid", line_color="black", line_width=2,
              annotation_text="Breakeven")

fig.update_layout(
    title='Profit Distribution by Scenario (n=10,000 each)',
    xaxis_title='Profit (100M KRW)',
    yaxis_title='Frequency',
    barmode='overlay',
    height=500, width=800,
    legend=dict(x=0.65, y=0.95)
)
fig.show()

# 시나리오별 결과 비교
print("📊 시나리오별 이익 분포:")
print(f"{'Scenario':<25} {'Mean':>8} {'Std':>8} {'5th':>8} {'95th':>8} {'P(>0)':>8}")
print("-" * 70)
for name, r in scenario_results.items():
    print(f"{name:<25} {r['mean']:>8.1f} {r['std']:>8.1f} {r['p5']:>8.1f} {r['p95']:>8.1f} {r['prob_positive']:>7.1f}%")

print(f"\n💡 시나리오에 따라 흑자 확률이 0%~100%로 극적으로 달라진다.")
print(f"   이는 '모든 시나리오에 대비'해야 함을 보여준다.")

In [ ]:
# 시나리오 가중 통합 분포
all_profits = []
weights = []

for name, params in scenarios.items():
    profit_data = scenario_results[name]['profit']
    n_draw = int(params['prob'] * n_sim)
    sampled = np.random.choice(profit_data, size=n_draw, replace=True)
    all_profits.extend(sampled)

combined = np.array(all_profits)

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=combined, nbinsx=60,
    marker_color='#72B7B2', opacity=0.75,
    name='Combined Distribution'
))
fig.add_vline(x=0, line_dash="solid", line_color="red", line_width=2,
              annotation_text="Breakeven")
fig.add_vline(x=np.mean(combined), line_dash="dash", line_color="green",
              annotation_text=f"Mean={np.mean(combined):.1f}")

fig.update_layout(
    title='Scenario-Weighted Combined Profit Distribution',
    xaxis_title='Profit (100M KRW)',
    yaxis_title='Frequency',
    height=400, width=700
)
fig.show()

# 통합 지표
print("📊 시나리오 가중 통합 결과:")
print(f"  기대 이익: {np.mean(combined):.1f}억 원")
print(f"  VaR(5%): {np.percentile(combined, 5):.1f}억 원")
print(f"  CVaR(5%): {np.mean(combined[combined <= np.percentile(combined, 5)]):.1f}억 원")
print(f"  흑자 확률: {np.mean(combined > 0)*100:.1f}%")

# 목표별 달성 확률
targets = [0, 10, 20, 30, 50]
print(f"\n📊 목표 달성 확률:")
for t in targets:
    print(f"  이익 > {t}억 원: {np.mean(combined > t)*100:.1f}%")

### 📝 이론 과제 11-3 (15분)

#### 과제: 시나리오-시뮬레이션 통합과 강건한 전략

**질문:**
1. **통합의 가치**: 시나리오 플래닝(10장)만 사용할 때와, 시뮬레이션을 결합할 때의 차이를 설명하시오. 시뮬레이션이 추가로 제공하는 정보는 무엇인가? (3문장)
2. **강건한 전략**: "기대이익이 가장 높은 전략"과 "최악의 시나리오에서도 손실이 작은 전략"이 다를 때, 어떤 기준으로 선택해야 하는가? 조직의 **리스크 선호도**와 연결하여 설명하시오. (3-4문장)
3. **상관관계**: 입력 변수 간 상관관계를 무시하면 시뮬레이션 결과가 어떻게 왜곡될 수 있는가? 예를 들어 설명하시오. (2-3문장)

**조사 키워드:**
- "robust strategy under uncertainty"
- "risk preference decision making"
- "correlated Monte Carlo simulation copula"

**제출:** 아래 마크다운 셀에 **직접 타이핑**

### ✅ 과제 11-3 제출란

**학번:** ___________
**이름:** ___________

#### 1. 시나리오 + 시뮬레이션 통합의 가치

(여기에 작성)

#### 2. 전략 선택과 리스크 선호도

(여기에 작성)

#### 3. 상관관계 무시의 문제

(여기에 작성)

---
# Part 4: 실습 — 사업 계획 시뮬레이션

**과제:** 신규 카페 창업(또는 본인이 선택한 사업)의 월간 수익을 시뮬레이션한다.

**요구사항:**
1. 입력 변수 최소 4개를 삼각분포로 정의한다
2. 10,000회 시뮬레이션을 실행한다
3. 이익 분포를 Plotly 히스토그램으로 시각화한다
4. 핵심 통계(평균, 90% CI, 흑자 확률)를 보고한다
5. 민감도 분석(토네이도 차트)을 수행한다

**카페 창업 예시 변수:**
- 일 평균 고객 수: 50~150명 (최빈 100명)
- 객단가: 4,000~8,000원 (최빈 6,000원)
- 원가율: 30~50% (최빈 40%)
- 월 임대료: 200~400만 원 (최빈 300만 원)
- 인건비: 300~500만 원 (최빈 400만 원)
- 영업일: 25~30일 (최빈 28일)

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

n_sim = 10000
np.random.seed(42)

# TODO 1: 입력 변수 정의 (삼각분포, 최소 4개)
# 힌트: np.random.triangular(최소, 최빈, 최대, n_sim)
daily_customers = np.random.triangular(50, 100, 150, n_sim)     # 일 평균 고객수
avg_spending =    # 여기에 코드 작성 (객단가, 원)
cost_ratio =      # 여기에 코드 작성 (원가율)
monthly_rent =    # 여기에 코드 작성 (월 임대료, 만원)
monthly_labor =   # 여기에 코드 작성 (월 인건비, 만원)
operating_days =  # 여기에 코드 작성 (월 영업일)

# TODO 2: 월 수익 모델 계산
monthly_revenue =   # 여기에 코드 작성 (일 고객수 × 객단가 × 영업일 / 10000, 만원 단위)
monthly_cogs =      # 여기에 코드 작성 (매출 × 원가율)
monthly_profit =    # 여기에 코드 작성 (매출 - 원가 - 임대료 - 인건비)

# TODO 3: 결과 시각화 (Plotly 히스토그램)
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=monthly_profit, nbinsx=60,
    marker_color='#4C78A8', opacity=0.75
))
fig.add_vline(x=0, line_dash="solid", line_color="red", line_width=2)
fig.add_vline(x=np.mean(monthly_profit), line_dash="dash", line_color="green")
fig.update_layout(
    title='Monthly Profit Distribution: Cafe Business',
    xaxis_title='Monthly Profit (10K KRW)',
    yaxis_title='Frequency',
    height=400, width=700
)
fig.show()

# TODO 4: 핵심 통계 출력
print(f"평균 월 이익: {np.mean(monthly_profit):.0f}만 원")
print(f"90% 신뢰구간: [{np.percentile(monthly_profit,5):.0f}, {np.percentile(monthly_profit,95):.0f}]만 원")
print(f"흑자 확률: {np.mean(monthly_profit > 0)*100:.1f}%")

# TODO 5: 민감도 분석 (토네이도 차트)
# 힌트: Part 2의 토네이도 차트 코드를 참고하되, 변수를 교체

# ========== 여기까지 ==========

---
# Part 5: 실습 — NPV 리스크 분석

**과제:** 5년간 투자 프로젝트의 NPV(순현재가치) 분포를 시뮬레이션한다.

**투자 조건:**
- 초기 투자: 100억 원
- 투자 기간: 5년
- 할인율: 10%
- 연간 현금흐름: 삼각분포 (년차별 다름)

**NPV 공식:**
NPV = -I₀ + Σ(CFₜ / (1+r)ᵗ)

여기서 I₀=초기투자, CFₜ=t년차 현금흐름, r=할인율

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

n_sim = 10000
np.random.seed(42)
initial_investment = 100  # 억 원
discount_rate = 0.10

# TODO 1: 연도별 현금흐름 정의 (삼각분포: 최소, 최빈, 최대)
cf_params = [
    (20, 30, 40),   # 1년차
    (25, 35, 45),   # 2년차
    (30, 40, 50),   # 3년차
    (25, 35, 45),   # 4년차
    (20, 30, 40),   # 5년차
]

# TODO 2: 현금흐름 시뮬레이션
cash_flows = np.zeros((n_sim, 5))
for year, (low, mode, high) in enumerate(cf_params):
    cash_flows[:, year] = np.random.triangular(low, mode, high, n_sim)

# TODO 3: NPV 계산
# 힌트: NPV = -초기투자 + Σ(CF_t / (1+r)^t)
npv_values = np.full(n_sim, -initial_investment, dtype=float)
for t in range(5):
    npv_values +=   # 여기에 코드 작성 (현금흐름 할인)

# TODO 4: NPV 분포 시각화
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=npv_values, nbinsx=60,
    marker_color='#F58518', opacity=0.75
))
fig.add_vline(x=0, line_dash="solid", line_color="red", line_width=2,
              annotation_text="Investment Threshold (NPV=0)")
fig.update_layout(
    title='NPV Distribution: 5-Year Investment Project',
    xaxis_title='NPV (100M KRW)',
    yaxis_title='Frequency',
    height=400, width=700
)
fig.show()

# TODO 5: 투자 의사결정 지표 출력
print(f"평균 NPV: {np.mean(npv_values):.1f}억 원")
print(f"VaR(5%): {np.percentile(npv_values, 5):.1f}억 원")
print(f"NPV > 0 확률: {np.mean(npv_values > 0)*100:.1f}%")

# 투자 권고
prob_positive = np.mean(npv_values > 0) * 100
if prob_positive >= 70:
    print(f"\n✅ 권고: 투자 추진 (성공 확률 {prob_positive:.0f}%)")
elif prob_positive >= 50:
    print(f"\n⚠️ 권고: 추가 검토 필요 (성공 확률 {prob_positive:.0f}%)")
else:
    print(f"\n❌ 권고: 투자 재고 (성공 확률 {prob_positive:.0f}%)")

# ========== 여기까지 ==========

---
## 🎓 강의 마무리 및 핵심 요약

### 오늘 배운 내용

1. **점 추정의 한계**: "매출 1억 원 예상"은 불확실성, 리스크, 목표 달성 확률을 숨긴다. 확률적 추정으로 전환해야 한다.
2. **확률 분포 선택**: 삼각분포(전문가 추정), 정규분포(데이터 충분), 로그정규분포(양수 값) 등 상황에 맞는 분포를 선택한다.
3. **시뮬레이션 결과 해석**: 평균, 90% 신뢰구간, VaR, 목표 달성 확률 등 다양한 관점에서 결과를 해석한다.
4. **민감도 분석**: 토네이도 차트로 핵심 리스크 변수를 식별하고 관리 우선순위를 결정한다.
5. **시나리오 통합**: 10장의 시나리오별로 시뮬레이션하고, 시나리오 확률로 가중하여 전체 리스크를 평가한다.

### 핵심 메시지
> 불확실성을 인정하고 정량화하라. **"대략 1억"**이라는 모호한 표현 대신 **"8천만~1.2억 원 (90% 확률)"**이라는 구체적 표현이 더 나은 의사결정을 가능하게 한다.

### 다음 장 예고
> **12장: 다기준 의사결정(MCDM)** — 시뮬레이션으로 불확실성을 정량화했다면, 이제 여러 대안 중 **어떤 것을 선택할지** 결정해야 한다. 상충하는 기준(비용, 품질, 리스크 등)을 체계적으로 조화시키는 방법을 학습한다.

### 📚 추가 학습 자료

**핵심 참고문헌:**
- Vose, D. (2008). Risk Analysis: A Quantitative Guide. Wiley.
- Hubbard, D.W. (2014). How to Measure Anything. Wiley.
- Raychaudhuri, S. (2008). Introduction to Monte Carlo Simulation. Winter Simulation Conference.

**Python 라이브러리:**
- `numpy`: 난수 생성, 통계 계산
- `plotly`: 인터랙티브 시각화
- `scipy.stats`: 고급 확률 분포

**실습 코드:**
- _전체 코드는 practice/chapter11/code/ 참고_